# Clasificación de Kidney Stone con YOLO11n-CLS

Notebook robusto para:
- reutilizar `train`, `val` y `test` si ya existen
- no volver a descargar `yolo11n-cls.pt` si ya existe localmente
- detectar automáticamente CUDA/GPU
- entrenar solo si todavía no existe `best.pt`
- evaluar el modelo con `test`


In [ ]:
# ============================================================
# 1. CONFIGURACIÓN
# ============================================================

from pathlib import Path

DATASET_ROOT = Path(r"D:\EXTRACTED\axial\axial\Kindy Stone Dataset")
ORIGINAL_DIR = DATASET_ROOT / "Original Dataset"
AUGMENTED_DIR = DATASET_ROOT / "Augmented Dataset"

TRAIN_DIR = DATASET_ROOT / "train"
VAL_DIR = DATASET_ROOT / "val"
TEST_DIR = DATASET_ROOT / "test"

MODEL_PATH = Path(r"D:\yolo11n-cls.pt")
RUNS_DIR = DATASET_ROOT / "runs"
RUN_NAME = "yolo11n_cls"

CLASS_NAMES = ["Non-Stone", "Stone"]

print("Dataset:", DATASET_ROOT)
print("Original:", ORIGINAL_DIR)
print("Train:", TRAIN_DIR)
print("Val:", VAL_DIR)
print("Test:", TEST_DIR)


In [ ]:
# ============================================================
# 2. COMPROBAR ESTRUCTURA
# ============================================================

required_dirs = [
    ORIGINAL_DIR / "Stone",
    ORIGINAL_DIR / "Non-Stone",
    TRAIN_DIR / "Stone",
    TRAIN_DIR / "Non-Stone",
    VAL_DIR / "Stone",
    VAL_DIR / "Non-Stone",
    TEST_DIR / "Stone",
    TEST_DIR / "Non-Stone",
]

missing = [str(p) for p in required_dirs if not p.exists()]

if missing:
    raise FileNotFoundError(
        "Faltan carpetas necesarias:\n" + "\n".join(missing)
    )

print("Estructura correcta.")


In [ ]:
# ============================================================
# 3. IMPORTS Y COMPROBACIÓN DE PYTORCH/CUDA
# ============================================================

import sys
import torch
import pandas as pd
import matplotlib.pyplot as plt

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
print("CUDA PyTorch:", torch.version.cuda)

if torch.cuda.is_available():
    DEVICE = 0
    print("GPU:", torch.cuda.get_device_name(0))
else:
    DEVICE = "cpu"
    print("GPU: NO DISPONIBLE -> se usará CPU")

print("Device seleccionado:", DEVICE)


In [ ]:
# ============================================================
# 4. CONTAR IMÁGENES DEL DATASET ORIGINAL
# ============================================================

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def listar_imagenes(carpeta):
    return sorted(
        p for p in carpeta.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    )

imagenes_stone = listar_imagenes(ORIGINAL_DIR / "Stone")
imagenes_non_stone = listar_imagenes(ORIGINAL_DIR / "Non-Stone")

print("Original - Stone:", len(imagenes_stone))
print("Original - Non-Stone:", len(imagenes_non_stone))
print("Total original:", len(imagenes_stone) + len(imagenes_non_stone))


In [ ]:
# ============================================================
# 5. VISUALIZAR EJEMPLOS
# ============================================================

from PIL import Image

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, imagen, titulo in [
    (axes[0], imagenes_stone[0], "Stone"),
    (axes[1], imagenes_non_stone[0], "Non-Stone"),
]:
    ax.imshow(Image.open(imagen))
    ax.set_title(titulo)
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 6. CREAR DATAFRAME DEL DATASET ORIGINAL
# ============================================================

datos = (
    [[str(p), 1, "Stone"] for p in imagenes_stone]
    + [[str(p), 0, "Non-Stone"] for p in imagenes_non_stone]
)

df = pd.DataFrame(datos, columns=["ruta", "label", "clase"])

print(df.head())
print("\nDistribución:")
print(df["clase"].value_counts())


In [ ]:
# ============================================================
# 7. VERIFICAR SPLITS EXISTENTES
# ============================================================

def contar_clases(carpeta):
    return {
        clase: len(listar_imagenes(carpeta / clase))
        for clase in CLASS_NAMES
    }

split_counts = {
    "train": contar_clases(TRAIN_DIR),
    "val": contar_clases(VAL_DIR),
    "test": contar_clases(TEST_DIR),
}

print("Train:", split_counts["train"])
print("Val:", split_counts["val"])
print("Test:", split_counts["test"])

splits_existentes = all(
    all(v > 0 for v in split_counts[nombre].values())
    for nombre in ["train", "val", "test"]
)

print("\nSplits completos:", splits_existentes)


In [ ]:
# ============================================================
# 8. CREAR SPLITS SOLO SI NO EXISTEN
# ============================================================

from sklearn.model_selection import train_test_split
import shutil

if splits_existentes:
    print("Los splits ya existen. NO se vuelven a generar ni copiar.")
else:
    print("Los splits están incompletos. Se crearán usando 70/15/15.")

    train_df, temp_df = train_test_split(
        df,
        test_size=0.30,
        stratify=df["label"],
        random_state=42,
    )

    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.50,
        stratify=temp_df["label"],
        random_state=42,
    )

    split_data = [
        ("train", train_df),
        ("val", val_df),
        ("test", test_df),
    ]

    for split_name, split_df in split_data:
        for _, row in split_df.iterrows():
            clase = row["clase"]
            origen = Path(row["ruta"])
            destino_dir = DATASET_ROOT / split_name / clase
            destino_dir.mkdir(parents=True, exist_ok=True)
            destino = destino_dir / origen.name

            if destino.exists():
                continue

            shutil.copy2(origen, destino)

    print("Splits creados sin sobrescribir archivos existentes.")


In [ ]:
# ============================================================
# 9. COMPROBAR SPLITS FINALES
# ============================================================

split_counts = {
    "train": contar_clases(TRAIN_DIR),
    "val": contar_clases(VAL_DIR),
    "test": contar_clases(TEST_DIR),
}

print("Train:", split_counts["train"])
print("Val:", split_counts["val"])
print("Test:", split_counts["test"])

print(
    "\nTotales:",
    sum(split_counts["train"].values()),
    sum(split_counts["val"].values()),
    sum(split_counts["test"].values()),
)


In [ ]:
# ============================================================
# 10. LOCALIZAR O DESCARGAR YOLO11n-CLS
# ============================================================

import requests

candidatos_modelo = [
    Path(r"D:\yolo11n-cls.pt"),
    DATASET_ROOT / "yolo11n-cls.pt",
    Path.cwd() / "yolo11n-cls.pt",
]

modelo_existente = next((p for p in candidatos_modelo if p.exists()), None)

if modelo_existente is None:
    MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

    url = "https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo11n-cls.pt"

    print("Modelo no encontrado. Descargando...")
    try:
        response = requests.get(url, stream=True, timeout=120)
        response.raise_for_status()

        with open(MODEL_PATH, "wb") as f:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)

        modelo_existente = MODEL_PATH
        print("Modelo descargado:", modelo_existente)

    except Exception as e:
        raise RuntimeError(
            "No se pudo descargar el modelo. Coloca manualmente "
            f"yolo11n-cls.pt en {MODEL_PATH}"
        ) from e
else:
    print("Modelo existente encontrado. NO se vuelve a descargar:")
    print(modelo_existente)

MODEL_PATH = modelo_existente


In [ ]:
# ============================================================
# 11. CARGAR YOLO11n-CLS
# ============================================================

from ultralytics import YOLO

modelo = YOLO(str(MODEL_PATH))

print("Modelo cargado correctamente:")
print(MODEL_PATH)


In [ ]:
# ============================================================
# 12. ENTRENAR SOLO SI NO EXISTE BEST.PT
# ============================================================

BEST_PT = RUNS_DIR / RUN_NAME / "weights" / "best.pt"

if BEST_PT.exists():
    print("Ya existe best.pt.")
    print("NO se vuelve a entrenar:")
    print(BEST_PT)

    modelo = YOLO(str(BEST_PT))
else:
    if DEVICE == "cpu":
        print("ADVERTENCIA: CUDA no está disponible. El entrenamiento será en CPU.")

    resultados = modelo.train(
        data=str(DATASET_ROOT),
        epochs=50,
        imgsz=224,
        batch=16,
        device=DEVICE,
        workers=0,
        patience=10,
        project=str(RUNS_DIR),
        name=RUN_NAME,
        exist_ok=True,
        pretrained=True,
        verbose=True,
    )

    print("Entrenamiento finalizado.")
    print("Best model:", BEST_PT)


In [ ]:
# ============================================================
# 13. VALIDACIÓN EN TEST
# ============================================================

from pathlib import Path

if not BEST_PT.exists():
    print("Se usará el modelo actualmente cargado.")
else:
    modelo = YOLO(str(BEST_PT))

metricas = modelo.val(
    data=str(DATASET_ROOT),
    split="test",
    imgsz=224,
    batch=16,
    device=DEVICE,
    workers=0,
)

print("Evaluación en test finalizada.")


In [ ]:
# ============================================================
# 14. MOSTRAR MÉTRICAS DE CLASIFICACIÓN
# ============================================================

print("Accuracy:", getattr(metricas, "top1", "No disponible"))
print("Top-5 Accuracy:", getattr(metricas, "top5", "No disponible"))


In [ ]:
# ============================================================
# 15. PREDICCIÓN DE UNA IMAGEN
# ============================================================

imagenes_test = listar_imagenes(TEST_DIR)

imagen_prueba = imagenes_test[0]

pred = modelo.predict(
    source=str(imagen_prueba),
    imgsz=224,
    device=DEVICE,
    verbose=False,
)

resultado = pred[0]

indice = int(resultado.probs.top1)
confianza = float(resultado.probs.top1conf)
clase_predicha = resultado.names[indice]

print("Imagen:", imagen_prueba)
print("Clase predicha:", clase_predicha)
print("Confianza:", round(confianza, 4))
